ThickCurr Example: Eddy currents in a conducting torus {#doc_thickcurr_torus_ex}
==========
In this example we use the \ref OpenFUSIONToolkit.ThickCurr.ThickCurr "ThickCurr" python interface to model eddy currents in a solid conducting torus (major radius $R_0 = 1$ m, minor radius $a = 0.5$ m, $\eta = 7 \times 10^{-7} \Omega \cdot m$) at the center of a spherical domain (radius 3 m) bounded by a perfectly conducting wall, with the remainder of the domain treated as vacuum. We will:

 1. Compute the slowest decay times (L/R time constants) of the system with an eigenvalue solve
 2. Simulate the response of the torus to the sudden removal of a uniform vertical magnetic field, using flux loops and a magnetic probe to record the response
 3. Generate plot files of the eigenmodes and time-dependent fields for ParaView

This example follows a similar structure to \ref doc_thickcurr_slab_ex, see that example for additional discussion of the setup and options. Unlike that example, a p-multigrid preconditioner is used, which allows for larger problems than a direct solver.

\note This example takes approximately 20 minutes to run in serial and requires approximately 2.7 GB of memory, with most of the time spent in the eigenvalue solve.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize']=(6,4)
plt.rcParams['font.weight']='bold'
plt.rcParams['axes.labelweight']='bold'
plt.rcParams['lines.linewidth']=2
plt.rcParams['lines.markeredgewidth']=2
%matplotlib inline
%config InlineBackend.figure_format = "retina"

### Load ThickCurr library

To load the ThickCurr python module we need to tell python where to the module is located. This can be done either through the `PYTHONPATH` environment variable or within a script using `sys.path.append()` as below, where we look for the environement variable `OFT_ROOTPATH` to provide the path to where the OpenFUSIONToolkit is installed (`/Applications/OFT` on macOS).

For the sensors used below we also import sensor classes from \ref OpenFUSIONToolkit.ThinCurr.sensor "ThinCurr", which can be used with ThickCurr without modification.

In [ ]:
oft_python_path = os.getenv('OFT_ROOTPATH')
if oft_python_path is not None:
    sys.path.append(os.path.join(oft_python_path,'python'))
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.ThickCurr import ThickCurr
from OpenFUSIONToolkit.ThinCurr.sensor import Mirnov, circular_flux_loop, flux_loop

## Approximate decay time
The slowest decaying mode of this system corresponds to a toroidal current in the torus, which links the hole of the torus. While there is no simple analytic solution for this case, the decay time can be estimated from the inductance and resistance of a torus carrying a uniform toroidal current
$$ L \approx \mu_0 R_0 \left[ ln \left( \frac{8 R_0}{a} \right) - \frac{7}{4} \right], \; R = \frac{2 \eta R_0}{a^2} $$
As the eigenmode adjusts its current distribution to maximize the decay time, the true value is expected to be somewhat larger than this estimate, while the nearby conducting wall reduces the inductance (and decay time) slightly.

In [ ]:
mu0 = 4.E-7*np.pi
R0, a, eta = 1.0, 0.5, 7.E-7
L_est = mu0*R0*(np.log(8.0*R0/a)-7.0/4.0)
R_est = 2.0*eta*R0/a**2
print('Estimated decay time = {0:.1f} ms'.format(L_est/R_est*1.E3))

## Setup model

### Initialize ThickCurr object

In [ ]:
myOFT = OFT_env(nthreads=2)
tc_model = ThickCurr(myOFT)

### Load mesh and build model
The mesh for this example (`thickcurr_torus.h5`, 12180 cells) is generated using [GMSH](https://gmsh.info/) with the script `create_torus_mesh.py`, which requires the `gmsh` python package. A second-order (curved) mesh is generated to accurately represent the curved surfaces of the torus and outer boundary, so `grid_order=2` is used. The torus is region 2 in the mesh and the vacuum is region 1.

The XML input file (`oft_in.xml`) sets the resistivity of the torus and defines the preconditioner, which is a p-multigrid V-cycle (`type="mg"`) between the second-order (`order=2`) and linear FE levels on the same mesh, enabled with `pmg=True`. On the second-order level the smoother is two iterations of GMRES, preconditioned with block-Jacobi using 200 local blocks, each solved with a direct (LU) solver. On the linear (coarse) level a single block (`nlocal=1`) is used, which corresponds to a direct solve of the full coarse system. Compared to a direct solve of the full system, this approach greatly reduces the required memory and allows larger meshes to be used (eg. a direct solve on a 6337 cell mesh required ~6 GB). Typically, 7-15 GMRES iterations are required per time step.

```xml
<oft>
  <thickcurr>
    <region>
      <id>2</id>
      <eta>7.0E-7</eta>
    </region>
    <pre type="mg">
      <smoother direction="both">
        <solver type="gmres">
          <its>2</its>
          <nrits>2</nrits>
          <pre type="block_jacobi">
            <nlocal>200</nlocal>
            <solver type="lu"></solver>
          </pre>
        </solver>
      </smoother>
      <coarse>
        <solver type="gmres">
          <its>1</its>
          <nrits>1</nrits>
          <pre type="block_jacobi">
            <nlocal>1</nlocal>
            <solver type="lu"></solver>
          </pre>
        </solver>
      </coarse>
    </pre>
  </thickcurr>
</oft>
```

In [ ]:
dt = 1.E-2
tc_model.setup_model(mesh_file='thickcurr_torus.h5', grid_order=2, order=2, xml_filename='oft_in.xml', pmg=True, dt=dt, lin_tol=1.E-10)

## Compute decay times
By default, the eigenvalue solve uses the time step as a shift, which must be small enough to accurately resolve the time-dependent evolution. As the convergence of the eigenvalue solve is much faster with a shift comparable to the slowest decay time, a separate shift is passed here. The linear system and preconditioner are rebuilt for the eigenvalue solve and restored afterward for the time-dependent simulation. For this example, using `shift=0.1` instead of the time step (`dt=1.E-2`) reduces the cost of the eigenvalue solve by more than a factor of two.

The slowest mode, which links the hole of the torus, is followed by a set of nearly degenerate modes with $\tau \approx 81$ ms.

In [ ]:
tau_eigs = tc_model.get_eigs(4, shift=0.1, save_plot=True)
for i, tau in enumerate(tau_eigs):
    print('Mode {0}: tau = {1:.2f} ms'.format(i+1, tau*1.E3))

## Time-dependent simulation

### Initial condition
The initial uniform vertical field $B_0 \hat{z}$ is set using the vector potential
$$ A_0 = \frac{B_0}{2} \left( -y \hat{x} + x \hat{y} \right) $$
Note that this field has a non-zero tangential vector potential (and normal magnetic field) on the outer boundary, which would be held fixed by the perfectly conducting wall, representing an externally applied field. Instead, these components are set to zero (`zero_boundary=True`, the default), which corresponds to the external field being switched off at $t = 0$. The field in the vacuum region then relaxes rapidly, while the flux linked by the torus is initially conserved by the eddy currents induced in the torus, which then decay.

In [ ]:
B0 = 0.1   # Initial vertical magnetic field [T]
def A_init(pts):
    A = np.zeros(pts.shape)
    A[:,0] = -B0*pts[:,1]/2.0
    A[:,1] = B0*pts[:,0]/2.0
    return A

energy0 = tc_model.set_field(A_init)
print('Initial magnetic energy = {0:.4E} J'.format(energy0))

### Define sensors
We add two circular flux loops in the $z = 0$ plane, one in the hole of the torus ($R = 0.45$ m) and one between the torus and the outer wall ($R = 2$ m), and a magnetic probe measuring $B_z$ at the center of the torus. Flux loops are evaluated as $\oint A \cdot dl$, so the mesh does not need to conform to a surface bounded by the loop.

In [ ]:
sensors = [
    circular_flux_loop(0.45, 0.0, 'FLUX_HOLE'),
    circular_flux_loop(2.0, 0.0, 'FLUX_OUTER'),
    Mirnov([0.0,0.0,0.0], [0.0,0.0,1.0], 'BZ_CENTER', dx=None)
]
sensor_obj = tc_model.setup_sensors(sensors)
for name, val in zip(sensor_obj['names'], tc_model.eval_sensors(sensor_obj)):
    print('{0:10}: {1:.4E}'.format(name, val))

### Run simulation

In [ ]:
tc_model.run_td(nsteps=100, plot_freq=5, sensor_obj=sensor_obj)

### Plot sensor histories
Initially the flux through the hole of the torus decays slowly, as faster decaying modes redistribute the induced currents, after which the flux decays with the slowest decay time from the eigenvalue solve. With backward Euler time stepping the observed decay time is $\Delta t / ln(1 + \Delta t / \tau)$, see \ref doc_thickcurr_slab_ex. The flux through the outer loop changes sign as the flux returns between the torus and the wall.

In [ ]:
sensor_hist = tc_model.load_history('floops.hist')
time = sensor_hist['time']
tau_eff = dt/np.log(1.0+dt/tau_eigs[0])
flux = sensor_hist['FLUX_HOLE']
tau_fit = -(time[-1]-time[-11])/np.log(flux[-1]/flux[-11])
print('Late-time decay of hole flux = {0:.2f} ms (expected {1:.2f} ms)'.format(tau_fit*1.E3, tau_eff*1.E3))

fig, ax = plt.subplots(1,2,figsize=(10,4),tight_layout=True)
for name in ('FLUX_HOLE', 'FLUX_OUTER'):
    ax[0].plot(time*1.E3, sensor_hist[name]*1.E3, label=name)
ax[0].plot(time*1.E3, flux[-1]*np.exp(-(time-time[-1])/tau_eff)*1.E3, 'k--', label=r'$e^{-t/\tau}$')
ax[0].set_ylabel('Flux [mWb]')
ax[0].set_ylim(-20,80)
ax[1].semilogy(time*1.E3, sensor_hist['BZ_CENTER'], label='BZ_CENTER')
ax[1].semilogy(time*1.E3, sensor_hist['BZ_CENTER'][-1]*np.exp(-(time-time[-1])/tau_eff), 'k--', label=r'$e^{-t/\tau}$')
ax[1].set_ylabel('$B_z$ [T]')
for ax_tmp in ax:
    ax_tmp.grid(True)
    ax_tmp.legend()
    ax_tmp.set_xlabel('Time [ms]')

## Visualization in ParaView
As for \ref doc_thickcurr_slab_ex, plot files are produced for the time-dependent fields in the run directory and for the eigenmodes in the `eigs` subdirectory. The current density (`J`) shows the toroidal eddy currents induced in the torus by the removal of the vertical field.

In [ ]:
tc_model.plot_td(plot_freq=5)
td_plot, eig_plot = tc_model.build_XDMF()

## Running in parallel
The same calls can be used in a python script that is run with MPI (eg. `mpirun -np 4 python run_torus.py`), where the mesh is automatically decomposed across processes. In this case only the head process (`myOFT.head_proc`) should read output files or produce plots, and all processes must make the same calls to ThickCurr.